# Which Language Family Has the Most Speakers?

This page uses the same [TidyTuesday 2026-01-13 dataset](https://github.com/rfordatascience/tidytuesday/tree/main/data/2026/2026-01-13)
on popular languages of Africa. Each row is one language–country pair, with the
language's family and its number of native speakers.

**Question:** which family of languages has the highest density of speakers?

"Density" can mean two things, so we answer both:

1. **Total speakers:** which family has the most native speakers overall?
2. **Speakers per language:** which family has the most speakers *per language*?

```{note} Code cells are folded
:class: dropdown
This analysis is written in R (tidyverse and ggplot2). Click a code cell to see the code.
```

In [1]:
suppressPackageStartupMessages(library(tidyverse))
options(repr.plot.width = 8, repr.plot.height = 5)

## The data

In [2]:
raw <- read_csv("data/africa.csv", show_col_types = FALSE)

africa <- raw |>
    distinct() |>
    mutate(family = recode(family, "Afro-Asiatic" = "Afroasiatic"))

tibble(
    measure = c("Rows (raw)", "Exact duplicate rows", "Rows after cleaning",
                "Language families"),
    value = c(nrow(raw), sum(duplicated(raw)), nrow(africa), n_distinct(africa$family))
)

measure,value
<chr>,<int>
Rows (raw),796
Exact duplicate rows,34
Rows after cleaning,762
Language families,16


Two cleaning steps: drop exact duplicate rows, and merge the two spellings
of the same family (`Afroasiatic` and `Afro-Asiatic`).

## Count each language once

A language spoken in several countries appears once **per country**, and every one
of those rows repeats the language's full number of native speakers. Arabic, for
example, appears in 12 rows with 150 million speakers each:

In [3]:
africa |>
    filter(language == "Arabic") |>
    select(language, family, native_speakers, country) |>
    head(5)

language,family,native_speakers,country
<chr>,<chr>,<dbl>,<chr>
Arabic,Afroasiatic,1.5e+08,Algeria
Arabic,Afroasiatic,1.5e+08,Chad
Arabic,Afroasiatic,1.5e+08,Comoros
Arabic,Afroasiatic,1.5e+08,Djibouti
Arabic,Afroasiatic,1.5e+08,Egypt


Summing speakers over rows would count Arabic 12 times (1.8 billion speakers,
more than Africa's whole population). So we keep **one row per language** before
adding up speakers.

A few names are shared by *different* languages (for example, Tonga in Zimbabwe,
Mozambique and Malawi has a different speaker count in each). So we treat a
language as a unique combination of name, family and speaker count, rather than by
name alone.

In [4]:
languages <- africa |>
    distinct(language, family, native_speakers)

family_summary <- languages |>
    group_by(family) |>
    summarise(
        num_languages = n(),
        total_speakers = sum(native_speakers),
        speakers_per_language = total_speakers / num_languages,
        .groups = "drop"
    ) |>
    mutate(speaker_share = total_speakers / sum(total_speakers) * 100) |>
    arrange(desc(total_speakers))

cat("Languages after counting each one once:", nrow(languages), "\n")
cat("Total native speakers:", round(sum(languages$native_speakers) / 1e6), "million\n")

family_summary |>
    mutate(
        total_speakers_m = round(total_speakers / 1e6, 1),
        speakers_per_language_m = round(speakers_per_language / 1e6, 2),
        speaker_share = round(speaker_share, 1)
    ) |>
    select(family, num_languages, total_speakers_m, speaker_share, speakers_per_language_m)

Languages after counting each one once: 510 
Total native speakers: 896 million


family,num_languages,total_speakers_m,speaker_share,speakers_per_language_m
<chr>,<int>,<dbl>,<dbl>,<dbl>
Niger–Congo,389,457.4,51.1,1.18
Afroasiatic,21,332.9,37.2,15.85
Nilo-Saharan,70,44.8,5.0,0.64
Indo-European,4,25.4,2.8,6.34
Austronesian,1,18.0,2.0,18.00
Kongo-based,1,13.0,1.5,13.00
French,2,1.2,0.1,0.59
Portuguese,1,0.9,0.1,0.87
English,2,0.9,0.1,0.43


## 1. Total speakers by family

In [5]:
big_families <- family_summary |> filter(total_speakers >= 10e6)
top_total <- big_families$family[1]

p1 <- big_families |>
    ggplot(aes(x = reorder(family, total_speakers), y = total_speakers / 1e6,
               fill = family == top_total)) +
    geom_col() +
    geom_text(aes(label = paste0(round(speaker_share), "%")), hjust = -0.15, size = 3.5) +
    scale_fill_manual(values = c(`TRUE` = "#ff7f0e", `FALSE` = "#1f77b4"), guide = "none") +
    scale_y_continuous(expand = expansion(mult = c(0, 0.12))) +
    coord_flip() +
    labs(
        title = paste(top_total, "has the most native speakers"),
        subtitle = "Families with at least 10 million speakers; labels show share of all speakers",
        x = NULL,
        y = "Total native speakers (millions)"
    ) +
    theme_minimal()

ggsave("images/language_density.png", p1, width = 8, height = 5, dpi = 150, bg = "white")

```{figure} images/language_density.png
:name: fig-family-total
:alt: Horizontal bar chart of total native speakers by language family, with Niger–Congo the largest, followed by Afroasiatic
:width: 85%

Total native speakers by language family, counting each language once.
Niger–Congo has the most speakers, followed by Afroasiatic.
```

## 2. Speakers per language

In [6]:
multi <- family_summary |> filter(num_languages >= 2, total_speakers >= 10e6)
top_density <- multi |> arrange(desc(speakers_per_language)) |> pull(family) |> first()

p2 <- multi |>
    ggplot(aes(x = reorder(family, speakers_per_language), y = speakers_per_language / 1e6,
               fill = family == top_density)) +
    geom_col() +
    geom_text(aes(label = paste(num_languages, "languages")), hjust = -0.1, size = 3.5) +
    scale_fill_manual(values = c(`TRUE` = "#ff7f0e", `FALSE` = "#1f77b4"), guide = "none") +
    scale_y_continuous(expand = expansion(mult = c(0, 0.3))) +
    coord_flip() +
    labs(
        title = paste(top_density, "has the most speakers per language"),
        subtitle = "Families with at least 2 languages and 10 million speakers",
        x = NULL,
        y = "Average native speakers per language (millions)"
    ) +
    theme_minimal()

ggsave("images/language_density_per_language.png", p2, width = 8, height = 4,
       dpi = 150, bg = "white")

```{figure} images/language_density_per_language.png
:name: fig-family-per-language
:alt: Horizontal bar chart of average native speakers per language by family, with Afroasiatic far ahead of Indo-European, Niger–Congo and Nilo-Saharan
:width: 85%

Average native speakers per language, for families with at least two languages
and 10 million speakers. Afroasiatic has a few very large languages, while
Niger–Congo has hundreds of smaller ones.
```

## Results

- **By total speakers, Niger–Congo is the largest family.** It has about half of all
  native speakers in the dataset, ahead of Afroasiatic. See [](#fig-family-total).
- **By speakers per language, Afroasiatic is far ahead.** Its few languages
  (such as Arabic and Hausa) are very large, while Niger–Congo's speakers are
  spread over hundreds of smaller languages. See [](#fig-family-per-language).
- **The answer depends on the definition of "density",** so it should be stated
  explicitly.
- **Counting each language once matters.** Summing over language–country rows
  would wrongly put Afroasiatic first, because widely spoken languages like Arabic
  are repeated once per country.

```{warning} Caveats
- **Popular languages only.** The dataset covers about 500 of Africa's
  1,250–2,100+ languages, so totals are not a full census of speakers.
- **Speaker counts are rough.** Native-speaker numbers come from Wikipedia and are
  estimates of different ages and quality.
- **Some family labels are cleaning artifacts.** A few creoles are labelled by
  their source language (for example "French" or "English"), and one language
  has the family "Language"; these are tiny and don't change the ranking.
- **Single-language families** (such as Austronesian, which is just Malagasy) are
  left out of the speakers-per-language chart, since one language is not a
  meaningful average.
```